In [1]:
import os

def show_inputs(depth=4, max_files_per_dir=6):
    """Print what is attached under /kaggle/input - check it before trusting the rest."""
    root0 = os.environ.get("SAT_INPUT", "/kaggle/input")
    for root, dirs, files in os.walk(root0):
        dirs[:] = sorted(d for d in dirs if not d.startswith("."))
        level = root.replace(root0, "").count(os.sep)
        if level > depth:
            dirs[:] = []
            continue
        print("  " * level + root + "/")
        for fn in sorted(files)[:max_files_per_dir]:
            print("  " * level + "  " + fn)
        if len(files) > max_files_per_dir:
            print("  " * level + f"  ... +{len(files) - max_files_per_dir} more files")

show_inputs()

/kaggle/input/
  /kaggle/input/datasets/
    /kaggle/input/datasets/shohanmahmudshimul/
      /kaggle/input/datasets/shohanmahmudshimul/sat-ext/
        /kaggle/input/datasets/shohanmahmudshimul/sat-ext/sat_ext/
          README.md
          analysis.py
          audit.py
          config.py
          dataset.py
          dota.py
          ... +10 more files
      /kaggle/input/datasets/shohanmahmudshimul/sat-splits/
        /kaggle/input/datasets/shohanmahmudshimul/sat-splits/audit/
        /kaggle/input/datasets/shohanmahmudshimul/sat-splits/splits/
      /kaggle/input/datasets/shohanmahmudshimul/sat-splits-extra/
        /kaggle/input/datasets/shohanmahmudshimul/sat-splits-extra/sat_splits_extra/
      /kaggle/input/datasets/shohanmahmudshimul/satelite-coco/
        README.dataset.txt
        README.roboflow.txt
        /kaggle/input/datasets/shohanmahmudshimul/satelite-coco/test/
          1011_jpeg.rf.141ad0f26ab02f1bfee50847febecb85.jpg
          1016_jpeg.rf.0aef8e3664a12b1681da

In [2]:
import os, sys, warnings
warnings.filterwarnings("ignore")
INPUT_ROOT = os.environ.get("SAT_INPUT", "/kaggle/input")

def _find_code_root():
    for root, dirs, files in os.walk(INPUT_ROOT):
        if {"config.py", "audit.py", "metrics.py", "infer.py"} <= set(files):
            return root
    return None

CODE_ROOT = os.environ.get("SAT_CODE") or _find_code_root()
assert CODE_ROOT, "sat_ext code folder not found - Add Input -> your 'sat-ext' code Dataset"
sys.path.insert(0, CODE_ROOT)
print("CODE_ROOT:", CODE_ROOT)

import numpy as np, pandas as pd
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 60); pd.set_option("display.max_colwidth", 70)
import config as cfg, utils
utils.ensure_dirs(); utils.set_seed(cfg.SEED)
print("work dir:", cfg.WORK_DIR, "| smoke test:", cfg.SMOKE)

CODE_ROOT: /kaggle/input/datasets/shohanmahmudshimul/sat-ext/sat_ext
work dir: /kaggle/working | smoke test: False


In [3]:
KEY = "07J"            # preset - do not change
PILOT = False         # True = 1-epoch YOLO11n check of the whole GPU pipeline (tagged __pilot, ignored by the tables)
import plans, train
PLAN = plans.PILOT if PILOT else plans.PLANS[KEY]
EPOCHS = 1 if (PILOT or cfg.SMOKE) else None
TAG = "pilot" if PILOT else ""
if PILOT:
    print("PILOT: grouped / YOLO11n / seed 42, 1 epoch")
else:
    plans.show(KEY)
print("epochs:", EPOCHS or cfg.TRAIN_PARAMS["epochs"])

07J-train-native-yolo11s-and-yolo11m-640: 12 runs, ~10.5 GPU h (planning estimate)
  split=native_grouped model=YOLO11s    seed=42
  split=native_grouped model=YOLO11s    seed=43
  split=native_grouped model=YOLO11s    seed=44
  split=native_grouped model=YOLO11s-1280 seed=42
  split=native_grouped model=YOLO11s-1280 seed=43
  split=native_grouped model=YOLO11s-1280 seed=44
  split=native_grouped model=YOLO11m    seed=42
  split=native_grouped model=YOLO11m    seed=43
  split=native_grouped model=YOLO11m    seed=44
  split=native_tiles model=YOLO11s    seed=42
  split=native_tiles model=YOLO11s    seed=43
  split=native_tiles model=YOLO11s    seed=44
epochs: 50


In [4]:
utils.require_gpu()
assert utils.install_detection_stack(), "pip install failed - is Internet ON? (Settings -> Internet)"
print(utils.versions())

GPU: {'device': 'cuda:0', 'gpu': 'Tesla T4', 'n_gpu': 2, 'torch': '2.11.0+cu128'}
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
{'ultralytics': '8.4.37', 'sahi': '0.12.8', 'torch': '2.11.0+cu128', 'pycocotools': 'installed', 'numpy': '2.1.3', 'cv2': '4.14.0'}


In [5]:
done = train.run_plan(PLAN, epochs=EPOCHS, tag=TAG)

materialized 'native_grouped' (2c526ef767ce) -> /kaggle/working/data_native_grouped  images {'train': 374, 'val': 80, 'test': 82}
New https://pypi.org/project/ultralytics/8.4.183 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.37 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.1, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/data_native_grouped/data.yaml, degrees=45.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.5, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=N

2026-10-21 11:24:23,426 - sahi - WARNING - ignoring invalid prediction with bbox: [753.358642578125, 1080, 801.8329467773438, 1080] (ultralytics.py:278)
2026-10-21 11:24:23,427 - sahi - WARNING - ignoring invalid prediction with bbox: [738.26025390625, 1080, 779.623779296875, 1080] (ultralytics.py:278)


  native_grouped__YOLO11s__s43 sahi640   AP50=0.3948 AP=0.1622 AR50_micro=0.935 | conf>=0.25: dets=8014 TP=2187 FP=5827 R=0.776 | 186.4 ms/img


2026-10-21 11:25:05,727 - sahi - WARNING - ignoring invalid prediction with bbox: [753.358642578125, 1080, 801.8329467773438, 1080] (ultralytics.py:278)
2026-10-21 11:25:05,729 - sahi - WARNING - ignoring invalid prediction with bbox: [738.26025390625, 1080, 779.623779296875, 1080] (ultralytics.py:278)


  native_grouped__YOLO11s__s43 sahi480   AP50=0.3531 AP=0.1409 AR50_micro=0.896 | conf>=0.25: dets=8495 TP=2101 FP=6394 R=0.746 | 289.6 ms/img


...

  native_tiles__YOLO11s__s44   sahi640   AP50=0.7176 AP=0.3028 AR50_micro=0.792 | conf>=0.25: dets=3826 TP=2106 FP=1720 R=0.748 | 161.5 ms/img


2026-10-21 16:28:45,859 - sahi - WARNING - ignoring invalid prediction with bbox: [1146.1761474609375, 0, 1305.310546875, 0] (ultralytics.py:278)


  native_tiles__YOLO11s__s44   sahi480   AP50=0.6517 AP=0.2768 AR50_micro=0.714 | conf>=0.25: dets=3907 TP=2052 FP=1855 R=0.728 | 268.4 ms/img


2026-10-21 16:29:39,747 - sahi - WARNING - ignoring invalid prediction with bbox: [1146.1761474609375, 0, 1305.310546875, 0] (ultralytics.py:278)


  native_tiles__YOLO11s__s44   sahi320   AP50=0.5788 AP=0.2690 AR50_micro=0.714 | conf>=0.25: dets=4814 TP=2028 FP=2786 R=0.720 | 467.1 ms/img

=== session summary ===
  native_grouped__YOLO11s__s42   trained+evaluated
  native_grouped__YOLO11s__s43   trained+evaluated
  native_grouped__YOLO11s__s44   trained+evaluated
  native_grouped__YOLO11s-1280__s42 trained+evaluated
  native_grouped__YOLO11s-1280__s43 trained+evaluated
  native_grouped__YOLO11s-1280__s44 trained+evaluated
  native_grouped__YOLO11m__s42   trained+evaluated
  native_grouped__YOLO11m__s43   trained+evaluated
  native_grouped__YOLO11m__s44   trained+evaluated
  native_tiles__YOLO11s__s42     trained+evaluated
  native_tiles__YOLO11s__s43     trained+evaluated
  native_tiles__YOLO11s__s44     trained+evaluated
session time used: 5.42 h


In [6]:
import shutil, glob
for rid, status in done:
    ev = os.path.join(cfg.RESULTS_DIR, rid, "eval_test.json")
    if os.path.isfile(ev):
        d = utils.load_json(ev)
        u = d["ultralytics_test"]
        print(f"{rid:<30} test mAP50={u.get('mAP50', float('nan')):.4f} mAP50-95={u.get('mAP50_95', float('nan')):.4f} | " +
              " ".join(f"{m}:AP={v['coco'].get('AP_all', float('nan')):.3f}" for m, v in d["modes"].items()))
z = shutil.make_archive(os.path.join(cfg.WORK_DIR, f"results_{KEY}"), "zip", cfg.RESULTS_DIR)
print("\nsmall results zip (no weights):", z, f"{os.path.getsize(z) / 1e6:.1f} MB")
for f in glob.glob(os.path.join(cfg.WORK_DIR, "runs")) + glob.glob(os.path.join(cfg.WORK_DIR, "data_*")):
    shutil.rmtree(f, ignore_errors=True)       # Ultralytics run folders are duplicated in results/ + weights/

native_grouped__YOLO11s__s42   test mAP50=0.5781 mAP50-95=0.2829 | standard:AP=0.283 sahi640:AP=0.229 sahi480:AP=0.178 sahi320:AP=0.152
native_grouped__YOLO11s__s43   test mAP50=0.5709 mAP50-95=0.3002 | standard:AP=0.299 sahi640:AP=0.162 sahi480:AP=0.141 sahi320:AP=0.128
native_grouped__YOLO11s__s44   test mAP50=0.6838 mAP50-95=0.3587 | standard:AP=0.353 sahi640:AP=0.266 sahi480:AP=0.256 sahi320:AP=0.236
native_grouped__YOLO11s-1280__s42 test mAP50=0.6708 mAP50-95=0.3935 | standard:AP=0.389 sahi640:AP=0.250 sahi480:AP=0.259 sahi320:AP=0.258
native_grouped__YOLO11s-1280__s43 test mAP50=0.7638 mAP50-95=0.4981 | standard:AP=0.487 sahi640:AP=0.234 sahi480:AP=0.247 sahi320:AP=0.345
native_grouped__YOLO11s-1280__s44 test mAP50=0.8161 mAP50-95=0.5326 | standard:AP=0.536 sahi640:AP=0.294 sahi480:AP=0.320 sahi320:AP=0.337
native_grouped__YOLO11m__s42   test mAP50=0.5430 mAP50-95=0.2158 | standard:AP=0.220 sahi640:AP=0.177 sahi480:AP=0.188 sahi320:AP=0.171
native_grouped__YOLO11m__s43   test mAP